# Using the `simple_dynamic_portfolios` library with the paper's spec and data

This notebook rebuilds the SPY/AGG/GLD results of
[*Simple Dynamic Stock/Bond/Gold Portfolios*](https://stanford.edu/~boyd/papers/stock_bond_gold_portfolios.html)
step by step with the library's public API: prepared pandas inputs in,
labelled results out.

The steps:

1. Data: public downloads, and the prepared market inputs.
2. The strategic baseline and the strategy configuration.
3. Return forecasts: ridge regression and EWMA.
4. The risk model: the 11-day rolling sample covariance.
5. Backtests: Markowitz, Simple Markowitz, and the benchmarks.
6. Evaluation over 2006–2026 and forecast accuracy (the paper's Figure 7).
7. A look inside the decisions.
8. Variations the library makes easy: anchor radius, initial portfolio, a
   prepared covariance, a different volatility limit.

All data is public and needs no API keys. The first run downloads it into
`data/` (not tracked); later runs reuse those files.

In [1]:
import os
from pathlib import Path

# Work from the repository root, so relative paths resolve the same way everywhere.
for _dir in [Path.cwd(), *Path.cwd().parents]:
    if (_dir / "pyproject.toml").exists():
        os.chdir(_dir)
        break

DATA = Path(os.environ.get("SBG_DATA", "data"))  # download cache
REFRESH = False  # True downloads everything again

## 1. Data and market inputs

Three public sources, all as in the paper:

- **Yahoo Finance**: SPY, AGG, and GLD adjusted closes (total-return prices)
  and volumes, 1995–2025, on SPY's trading calendar. A second download runs
  through today, used only to evaluate forecasts of the last days of 2025.
- **FRED**: the effective federal funds rate (the cash return), core CPI
  (inflation), and Treasury yields, VIX, and the dollar index (features).
- **Kenneth French's data library**: eight daily equity factors (features).

Each download is saved to `DATA` the first time and read back afterwards.
Yahoo's adjusted closes are restated backwards after every distribution, and
FRED and French series are occasionally revised, so a fresh download can move
the results below slightly from the paper's.

In [2]:
import io
import re
import zipfile
from dataclasses import replace
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import yfinance as yf
from plotly.subplots import make_subplots

# Interactive figures in JupyterLab, Notebook 7+, and VS Code (stored once, as plotly JSON).
pio.renderers.default = "plotly_mimetype"
pio.templates.default = "plotly_white"
COLORS = {"SPY": "#1f77b4", "AGG": "#ff7f0e", "GLD": "#2ca02c", "Cash": "#d62728"}

import simple_dynamic_portfolios as sdp

In [3]:
TICKERS = ["SPY", "AGG", "GLD"]
START, END = "1995-01-01", "2026-01-01"  # END is exclusive


def cached(names, download):
    """Read the named frames from DATA, or download, save, and read them back."""
    paths = [DATA / f"{name}.parquet" for name in names]
    if REFRESH or not all(p.exists() for p in paths):
        DATA.mkdir(parents=True, exist_ok=True)
        for frame, path in zip(download(), paths, strict=True):
            frame.to_parquet(path)
        print("downloaded", ", ".join(names))
    return [pd.read_parquet(p) for p in paths]


def yahoo(end=END):
    """Adjusted closes and volumes on SPY trading days."""
    data = yf.download(TICKERS, start=START, end=end, auto_adjust=True, progress=False)
    days = data["Close"]["SPY"].dropna().index
    return data["Close"].loc[days], data["Volume"].loc[days]


def fred(series_id):
    """One FRED series from its public CSV endpoint (missing values are '.')."""
    url = (f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
           f"&cosd={START}&coed={END}")
    with urlopen(Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=60) as response:
        frame = pd.read_csv(io.BytesIO(response.read()), index_col=0, parse_dates=True,
                            na_values=".")
    return frame.iloc[:, 0].astype(float).rename(series_id)


def french(zip_filename):
    """One daily file from Kenneth French's data library, in decimal returns."""
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{zip_filename}"
    with urlopen(url, timeout=60) as response, zipfile.ZipFile(io.BytesIO(response.read())) as zf:
        raw = zf.read(zf.namelist()[0]).decode("latin1")
    # The header is the line just before the first row starting with YYYYMMDD.
    first = next(i for i, line in enumerate(raw.splitlines()) if re.match(r"^\s*\d{8}\s*,", line))
    frame = pd.read_csv(io.StringIO(raw), skiprows=first - 1)
    dates = frame.pop(frame.columns[0]).astype(str).str.strip()
    is_date = dates.str.match(r"^\d{8}$")  # drops footers and annual tables
    frame = frame[is_date].set_axis(pd.to_datetime(dates[is_date], format="%Y%m%d"))
    frame.columns = [c.strip().replace("-", "_").replace(" ", "_").lower() for c in frame.columns]
    return frame.astype(float) / 100.0  # the files are in percent

In [4]:
closes, volumes = cached(["closes", "volumes"], yahoo)
(closes_eval,) = cached(["closes_eval"], lambda: [yahoo(end=None)[0][TICKERS]])
days = closes.index


def cash_and_inflation():
    # Federal funds rate, percent a year -> daily decimal return.
    ffr = fred("DFF").to_frame("FFR").reindex(days).ffill()
    # Monthly core CPI -> daily gross inflation, each month's change spread over 21 days.
    cpi = fred("CPILFESL").to_frame("CPI")
    cpi = ((1 + cpi.pct_change()) ** (1 / 21)).reindex(days.union(cpi.index)).ffill()
    return (1 + ffr / 100) ** (1 / 252) - 1.0, cpi.reindex(days)


MACRO = {"DGS3MO": "y_3m", "DGS2": "y_2y", "DGS5": "y_5y", "DGS10": "y_10y", "DGS30": "y_30y",
         "DFII10": "real_y_10y", "T10YIE": "breakeven_10y", "VIXCLS": "vix",
         "DTWEXBGS": "dollar_index"}
FRENCH = {"F-F_Research_Data_5_Factors_2x3_daily_CSV.zip": ["mkt_rf", "smb", "hml", "rmw", "cma"],
          "F-F_Momentum_Factor_daily_CSV.zip": ["mom"],
          "F-F_ST_Reversal_Factor_daily_CSV.zip": ["st_rev"],
          "F-F_LT_Reversal_Factor_daily_CSV.zip": ["lt_rev"]}

ffr, cpi_econ = cached(["ffr", "cpi_econ"], cash_and_inflation)
(macro_raw,) = cached(["macro_raw"], lambda: [
    pd.DataFrame({name: fred(sid) for sid, name in MACRO.items()}).reindex(days).ffill()])
(fama_french_raw,) = cached(["fama_french_raw"], lambda: [
    pd.concat([french(f)[cols] for f, cols in FRENCH.items()], axis=1, join="inner")
    .sort_index().loc[START:END].reindex(days).ffill()])
print(f"{len(days)} trading days, {days[0].date()} to {days[-1].date()}; "
      f"evaluation closes to {closes_eval.index[-1].date()}")

7802 trading days, 1995-01-03 to 2025-12-31; evaluation closes to 2026-09-25


The library takes **total-return prices** (adjusted closes, dates × assets) and
**daily decimal cash returns**. The paper's cash return is the federal funds
rate converted to a daily return.

In [5]:
prices = closes
cash = ffr["FFR"]  # daily decimal returns, not annual rates

market = sdp.MarketInputs(total_return_prices=prices, cash_returns=cash)
print("Assets (canonical order):", market.assets.tolist())
print("First price per asset:", {a: str(prices[a].first_valid_index().date()) for a in prices})
print(f"Cash: {cash.index[0].date()} to {cash.index[-1].date()}, "
      f"mean {cash.mean() * 252:.2%} a year")

Assets (canonical order): ['AGG', 'GLD', 'SPY']
First price per asset: {'AGG': '2003-09-29', 'GLD': '2004-11-18', 'SPY': '1995-01-03'}
Cash: 1995-01-03 to 2025-12-31, mean 2.50% a year


Missing prices before an asset's inception are fine: they only affect model
warm-up. Inside the simulation interval the library requires a complete panel
and says so, rather than silently filling:

In [6]:
try:
    market.to_backtest_data("M", start="2004-01-01")
except ValueError as error:
    print(error)

A complete price panel is required during the simulation interval; first missing price per asset: {'GLD': datetime.date(2004, 1, 2)}. Start later, drop the asset, or use BacktestData.from_pandas for legacy missing-price handling.


## 2. Baseline and configuration

The baseline is the strategic 50/30/20 risky mix the optimizer is anchored to.
It is a risky-sleeve vector summing to one; cash is not an entry.

The paper's strategy solves, at each month end,

$$
\max_w\ \alpha_t^\top w + r_{\text{cash},21}(1-\mathbf 1^\top w)
  - \tfrac{s}{2}\|w - w_{\text{current}}\|_1
$$

subject to $w \ge 0$, $\mathbf 1^\top w \le 1$, ex-ante volatility at most 7%
a year, and the trust region $\|w - g a\|_1 \le \rho g$ around the baseline $a$
(with $g = \mathbf 1^\top w$ and $\rho = 1$). Each setting is a field of
`MarkowitzConfig`, in explicit units.

In [7]:
baseline = sdp.StaticBaseline(pd.Series({"SPY": 0.5, "AGG": 0.3, "GLD": 0.2}))

config = sdp.MarkowitzConfig(
    annual_vol_target=0.07,     # annualized; converted to daily risk units
    max_risky_exposure=1.0,     # no leverage; the rest is cash
    relative_l1_radius=1.0,     # the paper's anchor constraint
    rebalance_frequency="M",
    objective_horizon_days=21,  # horizon of the forecasts and cash return
    bid_ask_spread=5e-4,        # full spread, shared by optimizer and simulator
    solver="CLARABEL",
)

SIMULATION = {"start": "2005-01-01", "end": "2026-01-01"}
EVALUATION = {"start": "2006-01-01", "end": "2026-01-01"}
print(f"Daily volatility limit: {config.daily_vol_target:.5f}")

Daily volatility limit: 0.00441


## 3. Return forecasts

**Ridge ("Markowitz").** The ridge model regresses each asset's annualized
100-day forward return on a feature matrix, walk-forward. It uses only matured
targets, up to 512 trailing observations with a one-year half-life, and ridge
penalty 10. The library accepts *any* feature DataFrame on the price index.
The paper's design is below, in natural units (the ridge model scales
features itself):

- **ETF features**, named `{feature}__{ticker}`: trailing 63-, 126-, and
  252-day returns, a volatility-adjusted return, the log ratio of 21-day to
  252-day volatility, and smoothed log volume. As in the paper's code, the
  volatility-adjusted return is overwritten at each horizon, so only the
  252-day version survives.
- **External features**: Treasury yield levels, the real yield, breakeven
  inflation, the 3-month/2-year slope, a smoothed log VIX, a slow dollar
  index, and 63-day means of the eight French factors.

In [8]:
returns = closes.pct_change()
blocks = {}
for h in [63, 126, 252]:
    blocks[f"ret_{h}d"] = closes.pct_change(h)
    blocks["vol_adjusted_ret"] = closes.pct_change(h) / returns.rolling(h).std()
blocks["vol_21_vs_252"] = np.log(returns.rolling(21).std()) - np.log(returns.rolling(252).std())
for w in [21, 63, 126]:
    blocks[f"log volume_{w}d"] = np.log(volumes).rolling(w).mean()
etf_features = pd.concat(blocks, axis=1)
etf_features.columns = [f"{feature}__{ticker}" for feature, ticker in etf_features.columns]

ext_features = macro_raw[["y_3m", "y_2y", "y_5y", "y_10y", "y_30y", "real_y_10y", "breakeven_10y"]].copy()
ext_features["log_vix"] = np.log(macro_raw["vix"]).rolling(63).mean()
ext_features["dollar_index"] = macro_raw["dollar_index"].rolling(512).mean()
ext_features["slope_3m_2y"] = ext_features["y_2y"] - ext_features["y_3m"]
ff_features = fama_french_raw.add_prefix("ff_").rolling(63).mean()

features = pd.concat([ext_features, ff_features, etf_features], axis=1)
print(features.shape)
print(features.columns.tolist()[:8], "...")

(7802, 42)
['y_3m', 'y_2y', 'y_5y', 'y_10y', 'y_30y', 'real_y_10y', 'breakeven_10y', 'log_vix'] ...


In [9]:
ridge_alpha = sdp.RidgeAlpha(
    horizon_days=100, max_history=512, min_window=63, halflife=252, ridge=10.0,
    assets=("AGG", "SPY", "GLD"),
).forecast(prices, features)

ewma_alpha = sdp.EwmaAlpha(halflife=252, min_periods=63).forecast(prices)

print("Ridge forecasts are per", ridge_alpha.horizon_days, "days (annualized)")
print("EWMA forecasts are per", ewma_alpha.horizon_days, "day")

Ridge forecasts are per 252 days (annualized)
EWMA forecasts are per 1 day


Forecasts carry their units. The runner rescales them to the optimizer's
21-day horizon (linearly, as in the paper), because alpha competes directly
with the cash return and trading costs in the objective. Here are the 21-day
forecasts the optimizer will see:

In [10]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=["Ridge: expected 21-day return", "EWMA: expected 21-day return"])
for col, alpha in enumerate([ridge_alpha, ewma_alpha], start=1):
    weekly = alpha.rescale_to_horizon(21).values.loc["2006":"2025"].resample("W").last()
    for asset in ["SPY", "AGG", "GLD"]:
        fig.add_trace(go.Scatter(x=weekly.index, y=weekly[asset], name=asset, legendgroup=asset,
                                 showlegend=col == 1, line={"color": COLORS[asset], "width": 1}),
                      row=1, col=col)
fig.add_hline(y=0.0, line_color="grey", line_width=0.5)
fig.update_yaxes(tickformat=".1%")
fig.update_layout(height=350, margin={"t": 40, "b": 30}, hovermode="x unified")
fig.show()

## 4. Risk model

The paper's estimator is the sample covariance of the last 11 complete daily
returns (a rolling window, not a half-life), in factor form
$\Sigma_t = F_tF_t^\top + \operatorname{diag}(d_t^2)$ with a tiny residual.
The factor order SPY, AGG, GLD matches the paper's Cholesky ordering.

In [11]:
risk = sdp.RollingSampleCovariance(window=11, residual_std=1e-7,
                                  assets=("SPY", "AGG", "GLD")).estimate(prices)

daily_var = (risk.loadings**2).sum(axis=1).unstack() + risk.residual_std**2
annual_vol = np.sqrt(252 * daily_var)
monthly_vol = annual_vol.loc["2006":"2025"].resample("ME").last()[["SPY", "AGG", "GLD"]]
fig = px.line(monthly_vol, color_discrete_map=COLORS,
              title="Annualized volatility from the 11-day estimate (month end)")
fig.update_layout(height=350, yaxis_tickformat=".0%", yaxis_title=None, xaxis_title=None,
                  legend_title=None, hovermode="x unified")
fig.show()
print("Covariance on 2020-03-31 (annualized):")
print((252 * risk.covariance("2020-03-31")).round(4))

Covariance on 2020-03-31 (annualized):
asset     AGG     GLD     SPY
asset                        
AGG    0.0873  0.0372  0.0111
GLD    0.0372  0.1686  0.1334
SPY    0.0111  0.1334  0.5364


## 5. Backtests

`run_markowitz` aligns forecasts, risk, and baseline to the market's asset
order and decision dates, then runs the optimizer and simulator. The
benchmarks use `run_fixed_weight` (a periodically rebalanced constant mix) and
`run_vol_controlled_fixed_weight` (the mix scaled to 7% using its own 11-day
volatility).

In [12]:
results = {
    "Markowitz": sdp.run_markowitz(market, baseline, ridge_alpha, risk, config,
                                  name="Markowitz", **SIMULATION),
    "Simple Markowitz": sdp.run_markowitz(market, baseline, ewma_alpha, risk, config,
                                         name="Simple Markowitz", **SIMULATION),
}

mix_6040 = sdp.StaticBaseline(pd.Series({"SPY": 0.6, "AGG": 0.4}))
for asset in ["SPY", "AGG", "GLD"]:
    results[asset] = sdp.run_fixed_weight(market, pd.Series({asset: 1.0}),
                                         rebalance_frequency="Y", name=asset, **SIMULATION)
for label, mix in [("60/40", mix_6040), ("50/30/20", baseline)]:
    results[label] = sdp.run_fixed_weight(market, mix, rebalance_frequency="Y",
                                         name=label, **SIMULATION)
    results[f"{label} VC"] = sdp.run_vol_controlled_fixed_weight(
        market, mix, annual_vol_target=0.07, vol_window=11, rebalance_frequency="M",
        name=f"{label} VC", **SIMULATION)
print(list(results))

['Markowitz', 'Simple Markowitz', 'SPY', 'AGG', 'GLD', '60/40', '60/40 VC', '50/30/20', '50/30/20 VC']


## 6. Evaluation

Metrics are computed over 2006–2026. The cash reference and inflation series
are evaluation inputs, separate from the cash the strategy actually earned.
The paper's "Sharpe Ratio (FFR)" is the **compounded** excess CAGR divided by
volatility; the conventional arithmetic Sharpe ratio is available separately.

In [13]:
cpi = cpi_econ["CPI"] - 1.0  # daily inflation
order = ["SPY", "AGG", "GLD", "60/40", "50/30/20", "60/40 VC", "50/30/20 VC",
         "Markowitz", "Simple Markowitz"]
table = pd.DataFrame({
    name: results[name].evaluate(cash_reference=cash, inflation=cpi, **EVALUATION)
    for name in order
}).T
table["Arithmetic Sharpe"] = [
    sdp.arithmetic_sharpe(results[name].backtest.between(**EVALUATION).navs, cash)
    for name in order
]
columns = ["Return", "Volatility", "Sharpe Ratio (FFR)", "Arithmetic Sharpe",
           "Max Drawdown", "Turnover", "Consistency"]
table[columns].style.format("{:.3f}")

,Return,Volatility,Sharpe Ratio (FFR),Arithmetic Sharpe,Max Drawdown,Turnover,Consistency
SPY,0.108,0.194,0.463,0.540,0.552,0.000,0.383
AGG,0.031,0.053,0.265,0.290,0.184,0.000,0.339
GLD,0.106,0.179,0.488,0.558,0.456,0.000,0.320
60/40,0.081,0.113,0.557,0.597,0.337,0.034,0.638
50/30/20,0.091,0.103,0.701,0.729,0.271,0.040,0.686
60/40 VC,0.071,0.074,0.712,0.731,0.169,0.850,0.955
50/30/20 VC,0.078,0.073,0.816,0.829,0.159,0.790,0.747
Markowitz,0.116,0.090,1.081,1.077,0.181,2.845,1.304
Simple Markowitz,0.104,0.093,0.913,0.923,0.158,2.789,1.046


These are the paper's numbers. In the original repository the same calls
reproduce its saved backtests exactly, NAV for NAV.

In [14]:
navs = pd.DataFrame({name: results[name].backtest.between(**EVALUATION).navs
                     for name in ["Markowitz", "Simple Markowitz", "50/30/20", "60/40"]})
growth = (navs / navs.iloc[0]).resample("W").last()  # weekly is plenty at this scale
fig = px.line(growth, log_y=True, title="Growth of $1 (log scale, weekly), 2006–2026")
fig.update_traces(line_width=1.2)
fig.update_layout(height=600, yaxis_title=None, xaxis_title=None, legend_title=None,
                  yaxis_tickprefix="$", hovermode="x unified")
fig.show()

The same strategies as year-over-year returns: each week, the return over the
trailing 252 trading days. The first value is a year into the evaluation window.

In [15]:
yoy = navs.pct_change(252, fill_method=None).dropna().resample("W").last()
fig = px.line(yoy, title="Year-over-year return (trailing 252 trading days, weekly), 2007–2026")
fig.update_traces(line_width=1.2)
fig.add_hline(y=0.0, line_color="grey", line_width=0.8)
fig.update_layout(height=600, yaxis_title=None, xaxis_title=None, legend_title=None,
                  yaxis_tickformat=".0%", hovermode="x unified")
fig.show()

### Forecast accuracy (the paper's Figure 7)

How well do the forecasts point in the right direction? On each day, take the
cosine similarity between the three-asset forecast vector $lpha_t$ and the
realized 100-day forward-return vector $r_t^{(h)}$. It ignores scale, so the
forecasts' units don't matter. Then smooth it with a rolling 100-day mean. The
realized returns for the last 100 days of 2025 need prices from 2026, so these
come from the evaluation-only download. Annual averages of the daily
similarity match the paper's table to machine precision.

In [16]:
ASSETS = ["SPY", "AGG", "GLD"]
H = 100
realized = closes_eval[ASSETS].pct_change(H, fill_method=None).shift(-H)


def cosine_similarity(forecast: pd.DataFrame, target: pd.DataFrame) -> pd.Series:
    # Row-wise cosine similarity; NaN where either vector is missing or zero.
    f, r = forecast[ASSETS].align(target[ASSETS], join="inner", axis=0)
    dot = (f * r).sum(axis=1, skipna=False)
    norms = np.sqrt((f**2).sum(axis=1, skipna=False) * (r**2).sum(axis=1, skipna=False))
    return (dot / norms).where(norms > 0)


window = slice(EVALUATION["start"], EVALUATION["end"])
cosine = pd.DataFrame({
    "Markowitz": cosine_similarity(ridge_alpha.values.loc[window], realized.loc[window]),
    "Simple Markowitz": cosine_similarity(ewma_alpha.values.loc[window], realized.loc[window]),
})
rolling_cosine = cosine.rolling(100).mean()

print("Mean daily cosine similarity, 2006-2025:", cosine.mean().round(3).to_dict())

Mean daily cosine similarity, 2006-2025: {'Markowitz': 0.361, 'Simple Markowitz': 0.434}


In [17]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_cosine.index, y=rolling_cosine["Markowitz"], name="Markowitz",
                         line={"color": "black", "dash": "dash", "width": 1.3}))
fig.add_trace(go.Scatter(x=rolling_cosine.index, y=rolling_cosine["Simple Markowitz"],
                         name="Simple Markowitz", line={"color": "grey", "width": 1.3}))
fig.add_hline(y=0.0, line_dash="dash", line_color="grey", line_width=0.8)
fig.update_layout(
    title="Cosine similarity of the forecast and the realized 100-day forward return"
          "<br><sup>rolling 100-day mean (the paper's Figure 7)</sup>",
    height=420, xaxis_title="Year", yaxis={"title": "Cosine similarity", "range": [-1, 1]},
    legend={"x": 0.01, "y": 0.02, "bgcolor": "rgba(255,255,255,0.7)"}, hovermode="x unified",
)
fig.show()

## 7. Inside the decisions

Each result keeps the target weights at every decision date and per-decision
diagnostics (solver status, the cash return used, risky exposure), along with
the forecasts (already in optimizer units) and risk model it used. So the
constraints can be checked directly:

In [18]:
mkw = results["Markowitz"]
print(mkw.diagnostics["status"].value_counts().to_dict())

ex_ante_vol = pd.Series({
    date: np.sqrt(252 * w.to_numpy() @ mkw.risk.covariance(date).to_numpy() @ w.to_numpy())
    for date, w in mkw.decisions.iterrows()
})
anchor = baseline.aligned(market.assets)
gross = mkw.decisions.sum(axis=1)
trust_region = (mkw.decisions.sub(np.outer(gross, anchor)).abs().sum(axis=1) / gross)
print(f"max ex-ante volatility: {ex_ante_vol.max():.4f} (limit 0.07)")
print(f"max ||w - g a||_1 / g:  {trust_region.max():.4f} (radius 1.0)")
mkw.decisions.tail().style.format("{:.3f}")

{'optimal': 253}
max ex-ante volatility: 0.0700 (limit 0.07)
max ||w - g a||_1 / g:  1.0000 (radius 1.0)


Ticker,AGG,GLD,SPY
date,,,
2025-08-29 00:00:00,0.025,0.580,0.223
2025-09-30 00:00:00,0.000,0.337,0.511
2025-10-31 00:00:00,0.000,0.088,0.583
2025-11-28 00:00:00,0.136,0.401,0.036
2025-12-31 00:00:00,0.388,0.147,0.465


In [48]:
composition = mkw.backtest.between(**EVALUATION).composition.resample("ME").last()
fig = px.area(composition[["SPY", "AGG", "GLD", "Cash"]].clip(lower=0),
              color_discrete_map=COLORS, title="Markowitz allocation (month end)")
fig.update_traces(line_width=0)
fig.update_layout(height=550, yaxis_range=[0, 1], yaxis_tickformat=".0%", yaxis_title=None,
                  xaxis_title=None, legend_title=None, hovermode="x unified")
fig.show()

## 8. Variations

None of these need changes to the optimizer or simulator: they are different
inputs or configuration.

**Anchor radius.** $\rho = 0$ holds the baseline proportions (so it behaves
like volatility-controlled 50/30/20 with a cash decision), and $\rho = 1$ is the paper.

In [42]:
def summary(result):
    m = result.evaluate(cash_reference=cash, **EVALUATION)
    return m[["Return", "Volatility", "Sharpe Ratio (FFR)", "Max Drawdown", "Turnover"]]

radius_sweep = pd.DataFrame({
    rho: summary(sdp.run_markowitz(market, baseline, ridge_alpha, risk,
                                  replace(config, relative_l1_radius=rho), **SIMULATION))
    for rho in [0.0, 0.25, 0.5, 1.0]
}).T
radius_sweep.index = [f"rho = {rho:g}" for rho in radius_sweep.index]
radius_sweep.style.format("{:.3f}")

,Return,Volatility,Sharpe Ratio (FFR),Max Drawdown,Turnover
rho = 0,0.067,0.068,0.730,0.127,0.870
rho = 0.25,0.086,0.074,0.919,0.146,1.235
rho = 0.5,0.102,0.080,1.042,0.167,1.821
rho = 1,0.116,0.090,1.081,0.181,2.845


**Initial portfolio.** By default the simulation starts in cash, and the
paper's Markowitz stays there until mid-2005. The ridge forecasts are zero
during warm-up (they need 63 matured 100-day targets for all three assets, and
GLD only starts in November 2004), and the first solve's cash return is zero by
construction. That warm-up is why evaluation starts in 2006.

Starting instead from an existing 60/40 holding, the optimizer sees those
weights as current. It keeps them on day one, since trading costs money and
nothing favors cash yet. At the first month end, cash earns the federal funds
rate while the forecasts are still zero, so it sells to cash and pays the
spread to do so:

In [21]:
start_6040 = sdp.InitialPortfolio.from_values(
    pd.Series({"SPY": 600_000.0, "AGG": 400_000.0}), cash_value=0.0)
from_6040 = sdp.run_markowitz(market, baseline, ridge_alpha, risk, config,
                             initial_portfolio=start_6040, **SIMULATION)

first_decisions = mkw.decisions.index[:3]
pd.concat({
    "from cash": mkw.decisions.loc[first_decisions].assign(
        turnover=mkw.backtest.turnover.loc[first_decisions]),
    "from 60/40": from_6040.decisions.loc[first_decisions].assign(
        turnover=from_6040.backtest.turnover.loc[first_decisions]),
}).style.format("{:.3f}")

**A prepared covariance.** Any risk source can be supplied as full covariance
matrices indexed by `(date, asset)`. Here it is a 63-day-half-life EWMA
estimate computed with pandas. The library validates symmetry and positive
semidefiniteness and factors each matrix exactly, without adding residual
variance.

In [22]:
returns = prices.pct_change(fill_method=None).dropna()  # complete rows start with GLD, Nov 2004
# Risk must exist on every decision date, the first being 2005-01-03, so warm up for 21 days.
ewma_covariance = returns.ewm(halflife=63, min_periods=21).cov().dropna()
ewma_covariance.index.names = ["date", "asset"]

alt_risk = sdp.run_markowitz(market, baseline, ridge_alpha,
                            sdp.CovarianceHistory(ewma_covariance), config, **SIMULATION)
pd.DataFrame({"11-day rolling (paper)": summary(mkw),
              "63-day EWMA": summary(alt_risk)}).T.style.format("{:.3f}")

,Return,Volatility,Sharpe Ratio (FFR),Max Drawdown,Turnover
11-day rolling (paper),0.116,0.090,1.081,0.181,2.845
63-day EWMA,0.084,0.076,0.863,0.202,0.922


**A different volatility limit.** The strategy need not reach its limit; it
holds cash when forecasts don't beat the cash rate after costs.

In [23]:
vol_sweep = pd.DataFrame({
    f"{v:.0%}": summary(sdp.run_markowitz(market, baseline, ridge_alpha, risk,
                                         replace(config, annual_vol_target=v), **SIMULATION))
    for v in [0.05, 0.07, 0.10, 0.12]
}).T
vol_sweep.index.name = "vol limit"
vol_sweep.style.format("{:.3f}")

,Return,Volatility,Sharpe Ratio (FFR),Max Drawdown,Turnover
vol limit,,,,,
5%,0.094,0.071,1.066,0.165,2.687
7%,0.116,0.090,1.081,0.181,2.845
10%,0.135,0.111,1.043,0.205,2.545
12%,0.140,0.121,1.001,0.223,2.138


## Where next

- `examples/second_universe.py` uses the same API on a synthetic five-asset
  universe, with no paper data or ticker names.
- The [original repository](https://github.com/cvxgrp/simple-portfolio-code)
  has the sensitivity studies and statistical inference built on this code.